In [1]:
pip install -qU langgraph langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In a traditional linear script, you pass a string from one function to the next. In LangGraph, you pass a shared object called a State between your functions (which we call Nodes). Every time a Node executes, it reads the current State, performs its task (like calling an LLM or running a tool), and then returns an update to that State.

For conversational agents, this State is almost always a running list of messages.

Here is exactly how the flow works:
1. Input: A Node (like your LLM or a tool) receives the master AgentState as its only input.
2. Action: The Node does its job (e.g., the LLM generates text, or the calculator computes 2+2).
3. Output: The Node returns a small dictionary containing only its new contribution. For example, a tool node might return {"messages": ["The answer is 4"]}.
4. The Merge: LangGraph catches that return value. Because we set up that add_messages rule earlier, LangGraph knows not to overwrite the whole state. It takes that new message, appends it to the master AgentState block, and then hands the updated master block to the next Node.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

# agent state is that shared clipboard passed along the chain of steps/nodes
# typedDict enfores the structure of this clipboard
class AgentState(TypedDict):
    # This tells the graph that the value inside the messages key will be a list
    # The `add_messages` reducer merges new messages into the existing list
        # Normally, if a function returns a dictionary like {in"messages": ["Hello"]}, it would overwrite whatever was already in the dictionary. 
        # The add_messages "reducer" acts as a rule: it tells LangGraph, "When a node hands you a new message, do NOT overwrite the history. 
        # Instead, append the new message to the end of the existing list."
    messages: Annotated[list, add_messages]

print("AgentState defined successfully! We are ready to build.")

/Users/pierson.chu/Downloads/github/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


AgentState defined successfully! We are ready to build.


In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# 1. Initialize the local model
# Change "llama3.1" if you are using a different model. 
# We use temperature=0 to make it as deterministic and logical as possible.
llm = ChatOllama(model="llama3.1", temperature=0)

# 2. Quick sanity check to ensure VSCode can talk to your local Ollama instance
print("Testing connection to local Ollama...")
test_message = HumanMessage(content="Reply with exactly one word: Howdy!")
test_response = llm.invoke([test_message])
print(f"Model replied: {test_response.content}\n")

# 3. Define the Reasoning Node
def reasoning_node(state: AgentState):
    """This function acts as the 'brain' node in our graph."""
    # Read the current history from our state clipboard
    messages = state['messages']
    # Pass the entire history to the local LLM
    response = llm.invoke(messages)
    # Return a dictionary with the NEW message. 
    # LangGraph's add_messages reducer will append this to the master state.
    return {"messages": [response]}

print("Step 2 & 3 Complete: ChatOllama connected and reasoning node defined!")

/Users/pierson.chu/Downloads/github/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Testing connection to local Ollama...
Model replied: Howdy!

Step 2 & 3 Complete: ChatOllama connected and reasoning node defined!


In [5]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

# 1. Initialize the Graph Builder
# We pass our AgentState dictionary so the graph knows what the clipboard looks like
graph_builder = StateGraph(AgentState)
# What is a stategraph? it is a manager of the workflow
    # It acts as a Registry
    # When you call .add_edge(), StateGraph draws the map. It ensures that when the reasoning node finishes, 
        # the flow moves strictly to where you told it to go (in our current case, END)
    # Because we initialized it by passing in AgentState, StateGraph knows the exact shape your data must take.
    

# 2. Add our Node
# We give it a string name ("reasoning") and tell it which function to run
graph_builder.add_node("reasoning", reasoning_node)

# 3. Define the Edges (The flow)
# Start the graph by routing exactly to our reasoning node
graph_builder.add_edge(START, "reasoning")

# For now, after the reasoning node finishes, end the graph
graph_builder.add_edge("reasoning", END)

# 4. Compile the Graph
# This locks in the structure and makes it an executable application
agent = graph_builder.compile()

# 5. Let's test it end-to-end!
print("Graph compiled! Running a test interaction...\n")

# We pass in our initial state (the user's first message)
initial_state = {"messages": [("user", "What is the capital of France?")]}
print(f"--- Input from user: What is the capital of France? ---")
# Stream the output as the graph runs
for event in agent.stream(initial_state):
    for node_name, node_state in event.items():
        print(f"--- Output from node: {node_name} ---")
        # Print the very last message added to the state
        print(node_state['messages'][-1].content)

print("\nStep 4 Complete! You just built and ran your first LangGraph.")

Graph compiled! Running a test interaction...

--- Input from user: What is the capital of France? ---
--- Output from node: reasoning ---
The capital of France is Paris.

Step 4 Complete! You just built and ran your first LangGraph.
